In [1]:
import pandas as pd
import numpy as np
import random
import sklearn.manifold as manifold
import colorsys

import plotly.express as px

In [2]:
# Read distance matrix
# dist_mat = pd.read_csv("../output/handshape_distance_matrix.csv")
dist_mat = pd.read_csv("../output/handshape_distance_matrix_handskel.csv")

dist_mat.head()

,Name,1_1.FBX,1_1_Jitter.FBX,1_2-2.FBX,1_2-3.FBX,1_2.FBX,1_Curved_Jitter-2.FBX,1_Curved_Jitter-3.FBX,1_Curved_Jitter.FBX,3.FBX,...,M20260506_0604_260713_0.fbx_8,M20260506_0604_260713_0.fbx_9,M20260506_0604_260713_0.fbx_10,M20260506_0604_260713_0.fbx_11,M20260506_0604_260713_0.fbx_12,M20260506_0604_260713_0.fbx_13,M20260506_0604_260713_0.fbx_14,M20260506_0604_260713_0.fbx_15,M20260506_0604_260713_0.fbx_16,M20260506_0604_260713_0.fbx_17
0,1_1.FBX,0.00000,1.29154,1.700850,1.596630,1.47584,2.35544,1.553600,1.62458,5.38570,...,6.04173,4.91861,5.71563,5.26966,4.48650,4.86900,5.71476,4.55041,4.59883,6.15265
1,1_1_Jitter.FBX,1.29154,0.00000,2.363280,2.217570,2.37235,3.15161,2.195690,2.33210,6.23857,...,6.56947,5.43428,6.27716,5.68757,4.97631,5.18502,5.48280,4.91136,5.04519,6.65184
2,1_2-2.FBX,1.70085,2.36328,0.000000,0.485771,1.26979,1.02582,0.501303,1.34755,5.48031,...,6.76501,5.82437,6.41839,6.24105,3.34458,5.90541,6.72117,5.58300,3.45103,6.91825
3,1_2-3.FBX,1.59663,2.21757,0.485771,0.000000,1.14081,1.30048,0.352566,1.21294,5.74841,...,6.67437,5.75588,6.32854,6.17867,3.49161,5.86530,6.57244,5.54104,3.60417,6.82653
4,1_2.FBX,1.47584,2.37235,1.269790,1.140810,0.00000,1.85650,1.278640,1.00781,5.18883,...,5.76250,4.84035,5.35977,5.33519,4.24326,5.11052,6.71320,4.82476,4.37325,5.91196


In [3]:
# Top 10 most similar handshapes to 'fbx\M20241107_6263_260710_2.fbx_segment_2'

# Get the row for the specified handshape
# Get file name from index 2
file_name = dist_mat.index[2]
handshape_row = dist_mat.iloc[2]

# Get the indices of the 10 smallest distances
handshape_distances = handshape_row[1:].to_numpy()
similar_indices = handshape_distances.argsort()

# Get the names of the 10 most similar handshapes (excluding any names starting with 'fbx\')
total = 0
similar_handshape_names = []
for index in similar_indices:
    if total >= 10:
        break
    handshape_name = dist_mat.columns[index + 1]  # +1 to account for the 'Name' column
    if not handshape_name.startswith('fbx\\'):
        similar_handshape_names.append(handshape_name)
        total += 1

print("10 most similar handshapes to:")
for handshape in similar_handshape_names:
    print(handshape)

10 most similar handshapes to:
1_2-2.FBX
1_2-3.FBX
1_Curved_Jitter-3.FBX
M20260119_1078_260713_1.fbx_1
1_Curved_Jitter-2.FBX
M20260127_2139_260713_0.fbx_4
M20260127_2139_260713_0.fbx_1
M20260127_2139_260713_0.fbx_5
M20260119_1078_260713_1.fbx_2
Baby_C.FBX


In [4]:
# Get distance array
dist_mat = dist_mat.drop(columns=['Name'])
distance_array = dist_mat.to_numpy()

In [5]:
# # Make all distances 0 or higher
# distance_array += np.abs(np.min(distance_array))

In [6]:
# t-SNE on the distance matrix
random_num = random.randint(0, 10000)
# random_num = 8079  # This is the last one I used, so for reproducibility hardcoded
print("Using random seed:", random_num)
tsne_dist = manifold.TSNE(n_components=2, perplexity=50, learning_rate=200, init='random', metric='precomputed', method='barnes_hut', random_state=random_num)
X_tsne_dist = tsne_dist.fit_transform(distance_array)

Using random seed: 5244


c:\Users\jusbo\AppData\Local\Programs\Python\Python312\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] Het systeem kan het opgegeven bestand niet vinden
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\jusbo\AppData\Local\Programs\Python\Python312\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\jusbo\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\jusbo\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close

In [8]:
# Dataframe for the px plot
tsne_df = pd.DataFrame({
    'tsne_1': X_tsne_dist[:, 0],
    'tsne_2': X_tsne_dist[:, 1],
    'name': dist_mat.columns,
    'is_handshape': [name.endswith(".FBX") for name in dist_mat.columns]
})

# Plot
fig = px.scatter(
    tsne_df,
    x='tsne_1',
    y='tsne_2',
    color='is_handshape',
    hover_data=['name'],
    title="t-SNE Dimensionality Reduction of the Shape Features"
)

fig.update_layout(width=800, height=800)
fig.show()